# Regularization Study
This notebook performs experiments to find out the influence that different regularization parameters bring to the model.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split
from google.colab import drive



drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/Image Classifier/image"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Ensure deterministic convolution algorithms
    torch.backends.cudnn.benchmark = False

seed_everything(42) # Lock random seed for reproducibility

In [ ]:
from torch.utils.data import Dataset
class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        '''
        Custom dataset for fruit classification.
        Image label is inferred from filename prefix.
        '''
        self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]
        self.class_map = {"apple": 0, "banana": 1, "orange": 2, "mixed": 3}
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

         # Label parsing logic
        label = -1
        for cls, idx_val in self.class_map.items():
            if filename.lower().startswith(cls):
                label = idx_val
                break
        return image, label

In [ ]:
def get_loaders(root_dir, img_size, batch_size):
    transform = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor()])
    full_dataset = FruitDataset(root_dir, transform=transform)
    indices = list(range(len(full_dataset)))
    labels = [full_dataset[i][1] for i in indices]
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)
    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

In [ ]:
# 3-layer CNN model with regularization support
class RegFruitCNN(nn.Module):
    def __init__(self, num_classes=4, use_bn=False, dropout_rate=0.0):
        super(RegFruitCNN, self).__init__()
        layers = []
        in_channels = 3
        filters = [16, 32, 64] #3-layer CNN model

        for i in range(len(filters)):
            layers.append(nn.Conv2d(in_channels, filters[i], kernel_size=3, padding=1))
            if use_bn: layers.append(nn.BatchNorm2d(filters[i])) # R2:batch normalization
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            in_channels = filters[i]

        self.conv_section = nn.Sequential(*layers)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate) if dropout_rate > 0 else nn.Identity(), # R1:dropout/not dropout
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_section(x); x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1); return self.fc(x)

In [ ]:
# basic train & evaluate
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss, correct, total = 0, 0, 0
    for imgs, lbs in loader:
        imgs, lbs = imgs.to(device), lbs.to(device)
        outputs = model(imgs); loss = criterion(outputs, lbs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == lbs).sum().item(); total += lbs.size(0)
    return total_loss / len(loader), correct / total

def evaluate(model, loader, device):
    model.eval(); correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbs in loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            outputs = model(imgs); _, preds = torch.max(outputs, 1)
            correct += (preds == lbs).sum().item(); total += lbs.size(0)
    return correct / total

if 'reg_results' not in locals(): reg_results = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Regularization Experiment is ready.")

Regularization Experiment is ready.


32*32 without dropout and BN

In [ ]:
import copy
# Import the copy module
# ================= Manual Modification Area =================
CURRENT_SIZE = 32       # Select 32 or 128
USE_BN = False          # R2 Switch: True to use BatchNorm
DROPOUT_VAL = 0.0       # R1 Switch: 0.3 to use Dropout, 0.0 to disable
EPOCHS = 30
# ==================================================


# Generate tag for saving results
reg_tag = "R0_None"
if USE_BN: reg_tag = "R2_BN"
elif DROPOUT_VAL > 0: reg_tag = f"R1_Drop{DROPOUT_VAL}"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16) # Use the pre-defined get_loaders
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize model with regularization
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} ")

# 3. Evaluate and record results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"Finished! Peak Val Acc: {best_val_acc:.4f}, Test Acc: {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Operating Experiment: Size32_R0_None


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 1.3461 | Train Acc: 0.2917 | Val Acc: 0.6042 
Epoch [02/30] Train Loss: 1.2537 | Train Acc: 0.5000 | Val Acc: 0.3125 
Epoch [03/30] Train Loss: 1.0166 | Train Acc: 0.6458 | Val Acc: 0.7500 
Epoch [04/30] Train Loss: 0.7578 | Train Acc: 0.6875 | Val Acc: 0.7083 
Epoch [05/30] Train Loss: 0.5532 | Train Acc: 0.8177 | Val Acc: 0.8125 
Epoch [06/30] Train Loss: 0.4025 | Train Acc: 0.8490 | Val Acc: 0.8542 
Epoch [07/30] Train Loss: 0.4285 | Train Acc: 0.8594 | Val Acc: 0.8542 
Epoch [08/30] Train Loss: 0.2916 | Train Acc: 0.9010 | Val Acc: 0.8542 
Epoch [09/30] Train Loss: 0.2572 | Train Acc: 0.9062 | Val Acc: 0.8958 
Epoch [10/30] Train Loss: 0.2527 | Train Acc: 0.9010 | Val Acc: 0.8542 
Epoch [11/30] Train Loss: 0.2078 | Train Acc: 0.9115 | Val Acc: 0.8958 
Epoch [12/30] Train Loss: 0.1632 | Train Acc: 0.9427 | Val Acc: 0.8958 
Epoch [13/30] Train Loss: 0.1466 | Train Acc: 0.9427 | Val Acc: 0.8750 
Epoch [14/30] Train Loss: 0.1334 | Train Acc: 0.9271 | Val Acc: 

32*32 + dropout 0.3 + no BN

In [ ]:
import copy
# Import the copy module
# ================= Manual Modification Area =================
CURRENT_SIZE = 32       # Select 32 or 128
USE_BN = False          # R2 Switch: True to use BatchNorm
DROPOUT_VAL = 0.3       # R1 Switch: 0.3 to use Dropout, 0.0 to disable
EPOCHS = 30
# ==================================================


# Generate tag for saving results
reg_tag = "R0_None"
if USE_BN: reg_tag = "R2_BN"
elif DROPOUT_VAL > 0: reg_tag = f"R1_Drop{DROPOUT_VAL}"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16) # Use the pre-defined get_loaders
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize model with regularization
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} ")

# 3. Evaluate and record results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"Finished! Peak Val Acc: {best_val_acc:.4f}, Test Acc: {test_acc:.4f}")

Operating Experiment: Size32_R1_Drop0.3


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 1.3473 | Train Acc: 0.2865 | Val Acc: 0.6042 
Epoch [02/30] Train Loss: 1.2681 | Train Acc: 0.5156 | Val Acc: 0.3333 
Epoch [03/30] Train Loss: 1.0883 | Train Acc: 0.5625 | Val Acc: 0.7292 
Epoch [04/30] Train Loss: 0.8020 | Train Acc: 0.7448 | Val Acc: 0.6875 
Epoch [05/30] Train Loss: 0.5715 | Train Acc: 0.8125 | Val Acc: 0.8125 
Epoch [06/30] Train Loss: 0.4150 | Train Acc: 0.8490 | Val Acc: 0.8125 
Epoch [07/30] Train Loss: 0.4514 | Train Acc: 0.8281 | Val Acc: 0.8542 
Epoch [08/30] Train Loss: 0.3630 | Train Acc: 0.8802 | Val Acc: 0.8333 
Epoch [09/30] Train Loss: 0.2800 | Train Acc: 0.9115 | Val Acc: 0.8333 
Epoch [10/30] Train Loss: 0.2663 | Train Acc: 0.9010 | Val Acc: 0.8542 
Epoch [11/30] Train Loss: 0.2560 | Train Acc: 0.8906 | Val Acc: 0.8750 
Epoch [12/30] Train Loss: 0.2566 | Train Acc: 0.9219 | Val Acc: 0.8333 
Epoch [13/30] Train Loss: 0.1973 | Train Acc: 0.9167 | Val Acc: 0.8542 
Epoch [14/30] Train Loss: 0.1474 | Train Acc: 0.9375 | Val Acc: 

32*32 + dropout 0.0 + BN

In [ ]:
import copy
# Import the copy module
# ================= Manual Modification Area =================
CURRENT_SIZE = 32       # Select 32 or 128
USE_BN = True          # R2 Switch: True to use BatchNorm
DROPOUT_VAL = 0.0       # R1 Switch: 0.3 to use Dropout, 0.0 to disable
EPOCHS = 30
# ==================================================


# Generate tag for saving results
reg_tag = "R0_None"
if USE_BN: reg_tag = "R2_BN"
elif DROPOUT_VAL > 0: reg_tag = f"R1_Drop{DROPOUT_VAL}"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16) # Use the pre-defined get_loaders
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize model with regularization
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} ")

# 3. Evaluate and record results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"Finished! Peak Val Acc: {best_val_acc:.4f}, Test Acc: {test_acc:.4f}")

Operating Experiment: Size32_R2_BN


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 0.9827 | Train Acc: 0.5729 | Val Acc: 0.3125 
Epoch [02/30] Train Loss: 0.5006 | Train Acc: 0.8021 | Val Acc: 0.7708 
Epoch [03/30] Train Loss: 0.3004 | Train Acc: 0.8906 | Val Acc: 0.8542 
Epoch [04/30] Train Loss: 0.2814 | Train Acc: 0.8906 | Val Acc: 0.9375 
Epoch [05/30] Train Loss: 0.1380 | Train Acc: 0.9635 | Val Acc: 0.9167 
Epoch [06/30] Train Loss: 0.0953 | Train Acc: 0.9792 | Val Acc: 0.8542 
Epoch [07/30] Train Loss: 0.1670 | Train Acc: 0.9531 | Val Acc: 0.9167 
Epoch [08/30] Train Loss: 0.0742 | Train Acc: 0.9740 | Val Acc: 0.8958 
Epoch [09/30] Train Loss: 0.0980 | Train Acc: 0.9531 | Val Acc: 0.9167 
Epoch [10/30] Train Loss: 0.0440 | Train Acc: 0.9844 | Val Acc: 0.9583 
Epoch [11/30] Train Loss: 0.0540 | Train Acc: 0.9844 | Val Acc: 0.9167 
Epoch [12/30] Train Loss: 0.1213 | Train Acc: 0.9479 | Val Acc: 0.9167 
Epoch [13/30] Train Loss: 0.0747 | Train Acc: 0.9688 | Val Acc: 0.9167 
Epoch [14/30] Train Loss: 0.0698 | Train Acc: 0.9896 | Val Acc: 

128*128 + dropout 0.0 + no BN

In [ ]:
import copy
# Import the copy module
# ================= Manual Modification Area =================
CURRENT_SIZE = 128       # Select 32 or 128
USE_BN = False          # R2 Switch: True to use BatchNorm
DROPOUT_VAL = 0.0       # R1 Switch: 0.3 to use Dropout, 0.0 to disable
EPOCHS = 30
# ==================================================


# Generate tag for saving results
reg_tag = "R0_None"
if USE_BN: reg_tag = "R2_BN"
elif DROPOUT_VAL > 0: reg_tag = f"R1_Drop{DROPOUT_VAL}"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16) # Use the pre-defined get_loaders
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize model with regularization
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} ")

# 3. Evaluate and record results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"Finished! Peak Val Acc: {best_val_acc:.4f}, Test Acc: {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Operating Experiment: Size128_R0_None


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 1.3437 | Train Acc: 0.2760 | Val Acc: 0.6667 
Epoch [02/30] Train Loss: 1.2245 | Train Acc: 0.5677 | Val Acc: 0.3125 
Epoch [03/30] Train Loss: 0.9725 | Train Acc: 0.6510 | Val Acc: 0.7500 
Epoch [04/30] Train Loss: 0.6911 | Train Acc: 0.7240 | Val Acc: 0.7500 
Epoch [05/30] Train Loss: 0.4874 | Train Acc: 0.8229 | Val Acc: 0.8333 
Epoch [06/30] Train Loss: 0.3431 | Train Acc: 0.8646 | Val Acc: 0.8333 
Epoch [07/30] Train Loss: 0.4184 | Train Acc: 0.8385 | Val Acc: 0.8750 
Epoch [08/30] Train Loss: 0.2769 | Train Acc: 0.8906 | Val Acc: 0.8958 
Epoch [09/30] Train Loss: 0.2178 | Train Acc: 0.9323 | Val Acc: 0.9375 
Epoch [10/30] Train Loss: 0.1472 | Train Acc: 0.9688 | Val Acc: 0.9167 
Epoch [11/30] Train Loss: 0.1397 | Train Acc: 0.9583 | Val Acc: 0.9375 
Epoch [12/30] Train Loss: 0.1039 | Train Acc: 0.9635 | Val Acc: 0.9375 
Epoch [13/30] Train Loss: 0.1035 | Train Acc: 0.9635 | Val Acc: 0.9167 
Epoch [14/30] Train Loss: 0.0734 | Train Acc: 0.9792 | Val Acc: 

128*128 + dropout 0.3 + no BN

In [ ]:
import copy
# Import the copy module
# ================= Manual Modification Area =================
CURRENT_SIZE = 128       # Select 32 or 128
USE_BN = False          # R2 Switch: True to use BatchNorm
DROPOUT_VAL = 0.3       # R1 Switch: 0.3 to use Dropout, 0.0 to disable
EPOCHS = 30
# ==================================================


# Generate tag for saving results
reg_tag = "R0_None"
if USE_BN: reg_tag = "R2_BN"
elif DROPOUT_VAL > 0: reg_tag = f"R1_Drop{DROPOUT_VAL}"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16) # Use the pre-defined get_loaders
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize model with regularization
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} ")

# 3. Evaluate and record results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"Finished! Peak Val Acc: {best_val_acc:.4f}, Test Acc: {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Operating Experiment: Size128_R1_Drop0.5


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 1.3852 | Train Acc: 0.2348 | Val Acc: 0.4483 
Epoch [02/30] Train Loss: 1.3170 | Train Acc: 0.4217 | Val Acc: 0.4828 
Epoch [03/30] Train Loss: 0.9962 | Train Acc: 0.5826 | Val Acc: 0.6552 
Epoch [04/30] Train Loss: 0.7941 | Train Acc: 0.6826 | Val Acc: 0.7069 
Epoch [05/30] Train Loss: 0.6872 | Train Acc: 0.7435 | Val Acc: 0.7931 
Epoch [06/30] Train Loss: 0.5047 | Train Acc: 0.7913 | Val Acc: 0.7759 
Epoch [07/30] Train Loss: 0.4702 | Train Acc: 0.8261 | Val Acc: 0.8103 
Epoch [08/30] Train Loss: 0.4585 | Train Acc: 0.8391 | Val Acc: 0.7931 
Epoch [09/30] Train Loss: 0.3793 | Train Acc: 0.8652 | Val Acc: 0.7931 
Epoch [10/30] Train Loss: 0.2644 | Train Acc: 0.9043 | Val Acc: 0.7931 
Epoch [11/30] Train Loss: 0.2864 | Train Acc: 0.9261 | Val Acc: 0.8621 
Epoch [12/30] Train Loss: 0.2962 | Train Acc: 0.8913 | Val Acc: 0.9310 
Epoch [13/30] Train Loss: 0.2485 | Train Acc: 0.8870 | Val Acc: 0.8621 
Epoch [14/30] Train Loss: 0.2356 | Train Acc: 0.9261 | Val Acc: 

128*128 + dropout 0.0 + BN

In [ ]:
import copy
# Import the copy module
# ================= Manual Modification Area =================
CURRENT_SIZE = 128       # Select 32 or 128
USE_BN = True          # R2 Switch: True to use BatchNorm
DROPOUT_VAL = 0.0       # R1 Switch: 0.3 to use Dropout, 0.0 to disable
EPOCHS = 30
# ==================================================


# Generate tag for saving results
reg_tag = "R0_None"
if USE_BN: reg_tag = "R2_BN"
elif DROPOUT_VAL > 0: reg_tag = f"R1_Drop{DROPOUT_VAL}"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16) # Use the pre-defined get_loaders
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize model with regularization
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} ")

# 3. Evaluate and record results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"Finished! Peak Val Acc: {best_val_acc:.4f}, Test Acc: {test_acc:.4f}")

Operating Experiment: Size128_R2_BN


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 0.9523 | Train Acc: 0.5990 | Val Acc: 0.3125 
Epoch [02/30] Train Loss: 0.5002 | Train Acc: 0.8281 | Val Acc: 0.6667 
Epoch [03/30] Train Loss: 0.3380 | Train Acc: 0.8854 | Val Acc: 0.7708 
Epoch [04/30] Train Loss: 0.3184 | Train Acc: 0.8906 | Val Acc: 0.9375 
Epoch [05/30] Train Loss: 0.2160 | Train Acc: 0.9167 | Val Acc: 0.9167 
Epoch [06/30] Train Loss: 0.1544 | Train Acc: 0.9531 | Val Acc: 0.9375 
Epoch [07/30] Train Loss: 0.1990 | Train Acc: 0.9167 | Val Acc: 0.9375 
Epoch [08/30] Train Loss: 0.1562 | Train Acc: 0.9271 | Val Acc: 0.9583 
Epoch [09/30] Train Loss: 0.2014 | Train Acc: 0.9219 | Val Acc: 0.9583 
Epoch [10/30] Train Loss: 0.1016 | Train Acc: 0.9688 | Val Acc: 0.9583 
Epoch [11/30] Train Loss: 0.1785 | Train Acc: 0.9375 | Val Acc: 0.9375 
Epoch [12/30] Train Loss: 0.1307 | Train Acc: 0.9635 | Val Acc: 0.9375 
Epoch [13/30] Train Loss: 0.1316 | Train Acc: 0.9531 | Val Acc: 0.9375 
Epoch [14/30] Train Loss: 0.0889 | Train Acc: 0.9792 | Val Acc: 

In [ ]:
def plot_4_3_study(results):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    for key, data in sorted(results.items()):
        if "Size32" in key:
            ax1.plot(range(1, 31), data['history'], label=f"{key} (Test:{data['test_acc']:.3f})")
        if "Size128" in key:
            ax2.plot(range(1, 31), data['history'], label=f"{key} (Test:{data['test_acc']:.3f})")

    ax1.set_title("32x32 Regularization Study"); ax1.set_ylim(0.3, 1.0); ax1.legend()
    ax2.set_title("128x128 Regularization Study"); ax2.set_ylim(0.3, 1.0); ax2.legend()
    plt.show()

plot_4_3_study(reg_results)

In [ ]:
# ================= 4.4 Combined Regularization Experiment (BN + Dropout) =================
CURRENT_SIZE = 128
USE_BN = True            # Enable BN
DROPOUT_VAL = 0.3        # Enable Dropout (0.3)
EPOCHS = 30
# ===================================================================

# --- Fix the tag generation logic to ensure both BN and Dropout are displayed simultaneously ---
tags = []
if USE_BN: tags.append("BN")
if DROPOUT_VAL > 0: tags.append(f"Drop{DROPOUT_VAL}")
reg_tag = "_".join(tags) if tags else "None"
exp_id = f"Size{CURRENT_SIZE}_{reg_tag}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

# 1. Load data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Initialize the model (Key: RegFruitCNN natively supports receiving both use_bn and dropout_rate)
model = RegFruitCNN(use_bn=USE_BN, dropout_rate=DROPOUT_VAL).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_model_wts = None
history = []

print(f"🚀 Operating Experiment: {exp_id}")

for epoch in range(EPOCHS):
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    v_acc = evaluate(model, val_loader, device)
    history.append(v_acc)

    # Record the best performance moment
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f} (Best: {best_val_acc:.4f})")

# 3. Evaluate and record the final results
model.load_state_dict(best_model_wts)
test_acc = evaluate(model, test_loader, device)
reg_results[exp_id] = {'history': history, 'test_acc': test_acc, 'best_val': best_val_acc}

print(f"\n✨ {exp_id} Finished!")
print(f"Peak Val Acc: {best_val_acc:.4f} | Test Acc: {test_acc:.4f}")

🚀 Operating Experiment: Size128_BN_Drop0.3


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Train Loss: 1.0289 | Train Acc: 0.5729 | Val Acc: 0.2917 (Best: 0.2917)
Epoch [02/30] Train Loss: 0.5398 | Train Acc: 0.8125 | Val Acc: 0.6250 (Best: 0.6250)
Epoch [03/30] Train Loss: 0.3302 | Train Acc: 0.8854 | Val Acc: 0.8542 (Best: 0.8542)
Epoch [04/30] Train Loss: 0.3470 | Train Acc: 0.8594 | Val Acc: 0.9167 (Best: 0.9167)
Epoch [05/30] Train Loss: 0.2179 | Train Acc: 0.9375 | Val Acc: 0.9167 (Best: 0.9167)
Epoch [06/30] Train Loss: 0.1719 | Train Acc: 0.9479 | Val Acc: 0.9375 (Best: 0.9375)
Epoch [07/30] Train Loss: 0.2760 | Train Acc: 0.8906 | Val Acc: 0.9167 (Best: 0.9375)
Epoch [08/30] Train Loss: 0.1808 | Train Acc: 0.9323 | Val Acc: 0.8958 (Best: 0.9375)
Epoch [09/30] Train Loss: 0.1980 | Train Acc: 0.9375 | Val Acc: 0.9583 (Best: 0.9583)
Epoch [10/30] Train Loss: 0.1410 | Train Acc: 0.9479 | Val Acc: 0.9375 (Best: 0.9583)
Epoch [11/30] Train Loss: 0.2281 | Train Acc: 0.9219 | Val Acc: 0.9375 (Best: 0.9583)
Epoch [12/30] Train Loss: 0.1543 | Train Acc: 0.9531 |

Combining experiments resolution and regularization, while a 128x128 resolution carries a risk of overfitting, introducing Dropout (0.3) significantly mitigates the fluctuations and achieves a generalization level consistent with lower resolution. Considering the significant advantages of high resolution in preserving mixed class features and subsequent data augmentation, 128x128 was ultimately chosen as the input size, coupled with Dropout as the core regularization method.